# CattleClass — Google Colab Training Notebook

This notebook clones the `cattleclass-project` branch, prepares the data, trains the TensorFlow/Keras cattle-breed classifier, launches the photo demo, and evaluates the model.

**Runtime:** In Colab, choose **Runtime → Change runtime type → T4 GPU** (or another available GPU).

**Important:** Keep credentials in Colab Secrets, never in notebook code or Git. The optional Git push section uploads only `results/` to the shared repository.


## 1. Mount Google Drive
Drive stores the working data and checkpoints so they survive a Colab runtime reset.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

%env CATTLE_WORK_DIR=/content/drive/MyDrive/CattleClass-work

Mounted at /content/drive
env: CATTLE_WORK_DIR=/content/drive/MyDrive/CattleClass-work


## 2. Clone the project and install dependencies
Run this the first time in a fresh runtime. If the folder already exists, use the update cell below instead.

In [ ]:
%cd /content
!git clone -b cattleclass-project https://github.com/ShayanAhmed12/CattleClass.git
%cd /content/CattleClass
!pip install -r requirements.txt

/content
Cloning into 'CattleClass'...
remote: Enumerating objects: 499, done.
remote: Counting objects: 100% (499/499), done.
remote: Compressing objects: 100% (293/293), done.
remote: Total 499 (delta 298), reused 392 (delta 191), pack-reused 0 (from 0)
Receiving objects: 100% (499/499), 913.72 KiB | 13.05 MiB/s, done.
Resolving deltas: 100% (298/298), done.
/content/CattleClass


### Update an existing checkout
Use this cell instead of cloning again if `/content/CattleClass` already exists.

In [ ]:
!git -C /content/CattleClass pull --ff-only origin cattleclass-project
%cd /content/CattleClass

From https://github.com/ShayanAhmed12/CattleClass
 * branch            cattleclass-project -> FETCH_HEAD
Already up to date.
/content/CattleClass


## 3. Check the runtime
Confirm that TensorFlow can see the GPU before training.

In [ ]:
import tensorflow as tf

print("TensorFlow:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))

TensorFlow: 2.21.0
GPU devices: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## 4. Prepare the dataset
Run data preparation before training. This may take some time depending on the dataset and network.

In [ ]:
%cd /content/CattleClass
!python prepare_data.py

/content/CattleClass
[cid] OK
[cid] 1893 images cached (119 removed after indexing)


## 5. Train the five breed folds
This trains the breed-classification folds. Completed compatible checkpoints may be skipped by the project training script.

In [ ]:
!python train.py

Fold 0: checkpoint verified, skipping
Fold 1: checkpoint verified, skipping
Fold 2: checkpoint verified, skipping
Fold 3: checkpoint verified, skipping
Fold 4: checkpoint verified, skipping
E1: macro-F1 0.484 ± 0.045
E2: macro-F1 0.511 ± 0.064


## 6. Fit the final breed model
The project guide indicates that this saves `models/final.keras` under the configured work directory.

In [ ]:
!python train.py --final

Final model: checkpoint verified, skipping


## 7. Launch the photo demo
The Gradio share link is public while the cell is running. Only upload photos you have permission to use.

In [ ]:
!python demo.py --share

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://541eab054371ba9489.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
W0000 00:00:1795341499.728357   10964 gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was false.
Keyboard interruption in main thread... closing server.
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/gradio/blocks.py", line 3957, in block_thread
    time.sleep(0.1)
    ~~~~~~~~~~^^^^^
KeyboardInterrupt

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/content/CattleClass/demo.py", line 113, in <module>
    main()
    ~~~~^^
  File "/content

### Alternative: predict directly in the notebook
Use this if the Gradio share link does not work. Upload a clear photo of one animal.

In [ ]:
from google.colab import files
from demo import predict

uploaded = files.upload()
if uploaded:
    image_path = next(iter(uploaded))
    print(predict(image_path))

Saving 267-rc-pro-1-1721024491.jpg to 267-rc-pro-1-1721024491.jpg
{'Sindhi': 0.5715170502662659, 'Red Chittagong': 0.31343454122543335, 'Sahiwal': 0.10148098319768906}


## 8. Evaluate breed predictions
After the five folds finish, generate evaluation metrics, robustness plots, and example prediction visualizations.

In [ ]:
!python evaluate.py --robustness --figures

W0000 00:00:1795341628.447054   11582 gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was false.
CID held-out macro-F1: 0.511


## 9. Optional: age and weight benchmarks
Run these only if you also need to reproduce the experimental age and weight results.

In [ ]:
!python train.py --traits
!python evaluate.py --traits

CID traits study e1f8e6cfe120: 503 SKUs
weight fold 0: verified saved predictions
weight fold 1: verified saved predictions
weight fold 2: verified saved predictions
weight fold 3: verified saved predictions
weight fold 4: verified saved predictions
age fold 0: verified saved predictions
age fold 1: verified saved predictions
age fold 2: verified saved predictions
age fold 3: verified saved predictions
age fold 4: verified saved predictions
weight: {'n': 503, 'mae_kg': 21.689420483932345, 'rmse_kg': 33.37381913697929, 'median_error_kg': 16.001678466796875, 'bias_kg': -2.2505354227176007, 'within_10_percent': 0.6759443339960238}, CID gate passed: True
age: {'n': 498, 'mae_months': 1.7951807228915662, 'accuracy': 0.7008032128514057, 'support': [375, 123], 'recall': [0.832, 0.3008130081300813], 'confusion': [[312, 63], [86, 37]], 'balanced_accuracy': 0.5664065040650407}, CID gate passed: True


## 10. Inspect generated results
Check the output files before deciding what to commit. Do not commit datasets, credentials, temporary files, or large checkpoints unless the team explicitly wants them in Git.

In [ ]:
!find results -maxdepth 3 -type f -print 2>/dev/null | head -200
!git status --short

results/capture_diagnostics.json
results/traits_diagnostics.json
results/cowdatabase2_weight.json
results/traits_metrics.csv
results/metrics.csv
results/cid_e6_details.json
results/cid_cv_details.json
results/cid_folds.csv
results/viewpoint_and_detection_audit.json
results/figures/cid_5-way_confusion.png
results/figures/local/gradcam.png
results/figures/local/error_annotations.csv
results/figures/local/error_gallery.png
results/figures/robustness.png
results/traits_evaluation.json
 M results/cid_cv_details.json
 M results/cid_folds.csv
 M results/metrics.csv
 M results/traits_evaluation.json
 M results/traits_metrics.csv
?? .gradio/
?? 267-rc-pro-1-1721024491.jpg


In [ ]:
!pwd
!ls -lah
!find /content -maxdepth 3 -name "*.ipynb"

/content/CattleClass
total 784K
drwxr-xr-x 10 root root 4.0K Nov 22 10:00 .
drwxr-xr-x  1 root root 4.0K Nov 22 09:23 ..
-rw-r--r--  1 root root 658K Nov 22 10:00 267-rc-pro-1-1721024491.jpg
drwxr-xr-x  3 root root 4.0K Nov 22 09:23 cattle_breed
-rw-r--r--  1 root root 4.1K Nov 22 09:23 demo.py
drwxr-xr-x  2 root root 4.0K Nov 22 09:23 docs
-rw-r--r--  1 root root 9.2K Nov 22 09:23 evaluate_capture.py
-rw-r--r--  1 root root 6.7K Nov 22 09:23 evaluate_public_weight.py
-rw-r--r--  1 root root 8.5K Nov 22 09:23 evaluate.py
drwxr-xr-x  8 root root 4.0K Nov 22 10:10 .git
-rw-r--r--  1 root root   12 Nov 22 09:23 .gitattributes
drwxr-xr-x  3 root root 4.0K Nov 22 09:23 .github
-rw-r--r--  1 root root  382 Nov 22 09:23 .gitignore
drwxr-xr-x  2 root root 4.0K Nov 22 09:37 .gradio
-rw-r--r--  1 root root 1.6K Nov 22 09:23 prepare_data.py
drwxr-xr-x  2 root root 4.0K Nov 22 10:00 __pycache__
-rw-r--r--  1 root root   56 Nov 22 09:23 pytest.ini
-rw-r--r--  1 root root    5 Nov 22 09:23 .python-v

In [ ]:
!git add colab.ipynb
!git add results/cid_cv_details.json results/cid_folds.csv results/metrics.csv results/traits_evaluation.json results/traits_metrics.csv

fatal: pathspec 'colab.ipynb' did not match any files


## 11. Resume after a runtime reset

1. Reconnect to a GPU runtime.
2. Rerun Sections 1 and 2, using the same Drive work directory.
3. If data preparation already completed, continue from training.
4. Run evaluation after training completes.
5. Check `results/` and push only the intended outputs.

**Tip:** Keep the runtime connected while training or while using the Gradio demo. The demo share link stops when its cell/runtime stops.
